In [1]:
import os
import ast
import warnings
warnings.filterwarnings("ignore")
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size': 10})
from sklearn.model_selection import train_test_split
# Importing neuroimaging package(s)
import pydicom

In [2]:
import sys
sys.path.append('../src/')

%load_ext autoreload
%autoreload 2
# Importing our custom module(s)
import ct
import mri

In [4]:
dicom_dir = "/cluster/tufts/hugheslab/datasets/RSNA"

#with open(f"{dicom_dir}/meta_data.csv", "w") as f:
#    f.write("Patient ID,Study ID,Series ID,Slice ID,z\n")
#    for root, _, files in os.walk(f"{dicom_dir}/stage_2_train"):
#        for name in files:
#            if name.endswith(".dcm"):
#                ds = pydicom.dcmread(os.path.join(root, name), stop_before_pixels=True)
#                f.write(f"{ds.PatientID},{ds.StudyInstanceUID},{ds.SeriesInstanceUID},{ds.SOPInstanceUID},{ds.ImagePositionPatient[2]}\n")

meta_data_df = pd.read_csv(f"{dicom_dir}/meta_data.csv")
stage_2_train_df = pd.read_csv(f"{dicom_dir}/stage_2_train.csv")
stage_2_train_df = stage_2_train_df.drop_duplicates()
stage_2_train_df["Slice ID"] = stage_2_train_df["ID"].str.rsplit("_", n=1).str[0]
stage_2_train_df["Label Type"] = stage_2_train_df["ID"].str.rsplit("_", n=1).str[1].str.capitalize()
stage_2_train_df = stage_2_train_df.pivot(index="Slice ID", columns="Label Type", values="Label").rename_axis(None, axis=1).reset_index()
meta_data_df = pd.merge(meta_data_df, stage_2_train_df, on="Slice ID", how="inner")
print(meta_data_df.shape)
print(len(meta_data_df[["Patient ID", "Series ID"]].value_counts()))
meta_data_df.head()


(752803, 11)
21744


,Patient ID,Study ID,Series ID,Slice ID,z,Any,Epidural,Intraparenchymal,Intraventricular,Subarachnoid,Subdural
0,ID_f15c0eee,ID_30ea2b02d4,ID_0ab5820b2a,ID_000012eaf,77.970825,0,0,0,0,0,0
1,ID_eeaf99e7,ID_134d398b61,ID_5f8484c3e0,ID_000039fa0,62.720940,0,0,0,0,0,0
2,ID_18f2d431,ID_b5c26cda09,ID_203cd6ec46,ID_00005679d,-39.569000,0,0,0,0,0,0
3,ID_ce8a3cd2,ID_974735bf79,ID_3780d48b28,ID_00008ce3c,175.995344,0,0,0,0,0,0
4,ID_d278c67b,ID_8881b1c4b1,ID_84296c3845,ID_0000950d7,157.500000,0,0,0,0,0,0


In [5]:
splits_df = pd.read_csv(f"{dicom_dir}/splits.csv")
print(splits_df.shape)
print(len(splits_df[["bag_name"]].value_counts()))
splits_df.head()

(39750, 2)
1149


,bag_name,split
0,ID_00047d6503,train
1,ID_00047d6503,train
2,ID_00047d6503,train
3,ID_00047d6503,train
4,ID_00047d6503,train


In [6]:
subset_df = meta_data_df[meta_data_df["Study ID"].isin(splits_df.bag_name)]
print(subset_df.shape)
print(len(subset_df[["Study ID"]].value_counts()))
subset_df.head()

(39710, 11)
1149


,Patient ID,Study ID,Series ID,Slice ID,z,Any,Epidural,Intraparenchymal,Intraventricular,Subarachnoid,Subdural
7,ID_df70c823,ID_04ef429610,ID_245e16180c,ID_0000f1657,367.000000,0,0,0,0,0,0
36,ID_40e9e7d3,ID_050fb6fd74,ID_515e22993b,ID_00042829c,24.067000,1,0,0,1,0,0
68,ID_f4a496fe,ID_0bf3cffbe6,ID_3d1e00ebf0,ID_0006a4a73,193.300049,0,0,0,0,0,0
91,ID_4a5cab9b,ID_feac26ebe2,ID_8b77760729,ID_000854890,88.020454,0,0,0,0,0,0
94,ID_d212cb48,ID_059e3f1d01,ID_f15cc986b7,ID_00087a0c0,203.642000,0,0,0,0,0,0


In [11]:
temp_df = meta_data_df.sort_values(["Study ID", "z"], ascending=[True, True])
grouped_df = temp_df.groupby(["Patient ID", "Study ID", "Series ID"]).agg({"Slice ID": list, "z": list, "Any": list}).reset_index()
grouped_df["paths"] = grouped_df.apply(lambda row: [f"{dicom_dir}/stage_2_train/{slice_id}.dcm" for slice_id in row["Slice ID"]], axis=1)
#grouped_df.to_csv(f"{dicom_dir}/labels.csv", index=False)
grouped_df.head()

,Patient ID,Study ID,Series ID,Slice ID,z,Any,paths
0,ID_0002cd41,ID_66929e09d4,ID_e22a5534e6,"[ID_45785016b, ID_37f32aed2, ID_1b9de2922, ID_...","[35.968, 38.484, 41.0, 43.517, 46.033, 48.549,...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",[/cluster/tufts/hugheslab/datasets/RSNA/stage_...
1,ID_00054f3f,ID_8a449ae31b,ID_15c3dd58c7,"[ID_138d275c8, ID_447fa09d9, ID_0f1298f68, ID_...","[71.9000244, 76.9000244, 81.9000244, 86.900024...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",[/cluster/tufts/hugheslab/datasets/RSNA/stage_...
2,ID_0006d192,ID_25690b4725,ID_4ec55fa0d7,"[ID_c6f9f68c9, ID_520df89aa, ID_b86dc15dd, ID_...","[38.171, 41.921, 45.671, 49.421, 53.171, 56.92...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",[/cluster/tufts/hugheslab/datasets/RSNA/stage_...
3,ID_00086119,ID_fdde2979b0,ID_caa405a7f2,"[ID_31b14de96, ID_203ef1efe, ID_9ce17ada6, ID_...","[32.955, 35.556, 38.156, 40.757, 43.358, 45.95...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",[/cluster/tufts/hugheslab/datasets/RSNA/stage_...
4,ID_000e5623,ID_9a4be35b9a,ID_0880626a95,"[ID_0785539ea, ID_30c100dbc, ID_3df0d63c3, ID_...","[272.0, 277.0, 282.0, 287.0, 292.0, 297.0, 302...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",[/cluster/tufts/hugheslab/datasets/RSNA/stage_...


In [7]:
ds = pydicom.dcmread(f"{dicom_dir}/stage_2_train/ID_000012eaf.dcm")
ds

Dataset.file_meta -------------------------------
(0002, 0000) File Meta Information Group Length  UL: 188
(0002, 0001) File Meta Information Version       OB: b'\x00\x01'
(0002, 0002) Media Storage SOP Class UID         UI: CT Image Storage
(0002, 0003) Media Storage SOP Instance UID      UI: 1.2.840.4267.32.337944818669776895705763408052798539612
(0002, 0010) Transfer Syntax UID                 UI: Explicit VR Little Endian
(0002, 0012) Implementation Class UID            UI: 1.2.40.0.13.1.1.1
(0002, 0013) Implementation Version Name         SH: 'dcm4che-1.4.35'
-------------------------------------------------
(0008, 0018) SOP Instance UID                    UI: ID_000012eaf
(0008, 0060) Modality                            CS: 'CT'
(0010, 0020) Patient ID                          LO: 'ID_f15c0eee'
(0020, 000d) Study Instance UID                  UI: ID_30ea2b02d4
(0020, 000e) Series Instance UID                 UI: ID_0ab5820b2a
(0020, 0010) Study ID                            SH: '